# Train the Baseline CNN on a Free GPU

This notebook downloads the weather data, trains the 3-day and 5-day forecast networks, scores them on 2017 and 2018, and downloads the results as `cnn_results.zip`.

In the menu choose Runtime > Run all. It takes roughly 30 to 60 minutes and needs no input.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU found. In the menu choose Runtime > Change runtime type > T4 GPU, then Run all again."
)
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import os
import subprocess
import sys

REPO = "/content/weather-forecaster"
URL = "https://github.com/KairavT/weather-forecaster.git"
if not os.path.isdir(REPO):
    # the code lives on the first-model branch until its pull request is merged
    !git clone -q --branch first-model {URL} {REPO} || git clone -q {URL} {REPO}
%cd {REPO}
!pip install -q "zarr>=3" gcsfs certifi
os.environ["PYTHONPATH"] = f"{REPO}/src"


def run(*args):
    """Run one of the repo's scripts, show its output live, and stop the notebook if it fails."""
    process = subprocess.Popen(
        [sys.executable, "-u", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"{' '.join(args)} failed, see the output above")

Download three fields for 1979 to 2020 (about 1.5 GB) and check the no-model baselines.

In [ ]:
run("scripts/download.py", "--preset", "baselines")
run("scripts/baselines.py")

Train one network per lead time. Each stops early once the 2016 validation score stops improving.

In [ ]:
run("scripts/train.py", "--lead-days", "3", "--epochs", "30")

In [ ]:
run("scripts/train.py", "--lead-days", "5", "--epochs", "30")

Show the test scores next to the WeatherBench paper's CNN, then download everything as one zip.

In [ ]:
import json

from google.colab import files

PUBLISHED = {"cnn-3d": (626, 2.87), "cnn-5d": (757, 3.37)}  # Rasp et al. 2020, Table 2

with open("results/models.json") as f:
    scores = json.load(f)
for name, entry in scores.items():
    z500 = entry["test_rmse"]["geopotential_500"]
    t850 = entry["test_rmse"]["temperature_850"]
    paper_z500, paper_t850 = PUBLISHED[name]
    print(
        f"{name}: Z500 {z500:.0f} (paper {paper_z500}), T850 {t850:.2f} (paper {paper_t850}), "
        f"{entry['epochs_run']} epochs"
    )

!zip -q -r cnn_results.zip results/models.json runs
files.download("cnn_results.zip")